# ミニカーバトル２０２５走行（予選2025/10/25)

update 2025/10/25 12:48

4つの走行モデルをと環境モデルを用いて走行を切り替えます。

cam0: 走行用カメラ
cam1: 環境認識用カメラ

走行モデルは、信号機の左、直進、右のモデルと駐車モデルを読み込みし走行
環境モデルは、コース場所を特定

対応デバイス: Jetson Jetson Orin Nano

   "GATE_LEFT",  # 電子掲示板で左矢印表示

    "GATE_RIGHT",  # 電子掲示板で右矢印表示

    "GATE_CENTER",  # 電子掲示板で前矢印表示

    "PARKING",  # 駐車状態

    "ETC"  # 分類できないもの

## Jetsonの認識

In [ ]:
import os

# ---------- 1. Jetson.GPIO 読み取り ----------
try:
    import Jetson.GPIO as GPIO
    BOARD_NAME = GPIO.gpio_pin_data.get_data()[0]
except Exception as e:
    # 失敗したら Orin Nano と決め打ち
    print(f"[WARN] Jetson モデル判定エラー: {e} → 強制的に JETSON_ORIN_NANO として続行")
    os.environ["JETSON_MODEL_NAME"] = "JETSON_ORIN_NANO"
    import Jetson.GPIO as GPIO          # もう一度ロード
    BOARD_NAME = "JETSON_ORIN_NANO"     # 確定

# ---------- 2. ボード別定義 ----------
mode_descriptions = {
    "JETSON_NX":       ["15W_2CORE", "15W_4CORE", "15W_6CORE", "10W_2CORE", "10W_4CORE"],
    "JETSON_XAVIER":   ["MAXN", "MODE_10W", "MODE_15W", "MODE_30W"],
    "JETSON_NANO":     ["MAXN", "5W"],
    "JETSON_ORIN":     ["MAXN", "MODE_15W", "MODE_30W", "MODE_40W"],
    "JETSON_ORIN_NANO":["MODE_15W", "MODE_25W", "MODE_MAX"]
}

product_names = {
    "JETSON_NX":        "Jetson Xavier NX",
    "JETSON_XAVIER":    "Jetson AGX Xavier",
    "JETSON_NANO":      "Jetson Nano",
    "JETSON_ORIN":      "Jetson AGX Orin",
    "JETSON_ORIN_NANO": "Jetson Orin Nano"
}

# (I2C バス番号, 初期 Power モードインデックス)
board_settings = {
    "JETSON_NX":        (8, 3),
    "JETSON_XAVIER":    (8, 2),
    "JETSON_NANO":      (1, 0),
    "JETSON_ORIN":      (7, 0),
    "JETSON_ORIN_NANO": (7, 2)
}

# ---------- 3. パラメータ取得 ----------
i2c_busnum, power_mode = board_settings.get(BOARD_NAME, (None, None))
mode_list       = mode_descriptions.get(BOARD_NAME, [])
product_name    = product_names.get(BOARD_NAME, "未知のボード")

# ---------- 4. 出力 ----------
if i2c_busnum is not None and 0 <= power_mode < len(mode_list):
    mode_str = mode_list[power_mode]
    print("------------------------------------------------------------")
    print(f"{product_name} を認識: I2C バス番号 = {i2c_busnum}, "
          f"Power モード = {mode_str} ({power_mode})")
    print("------------------------------------------------------------")
else:
    raise RuntimeError(f"未対応の Jetson モデル、または Power モード定義不足: {BOARD_NAME}")

In [ ]:
!echo "jetson" | sudo -S nvpmodel -m $power_mode

In [ ]:
!echo "jetson" | sudo -S nvpmodel -q

In [ ]:
!echo "jetson" | sudo -S jetson_clocks

## ログの表示用 Widget

In [ ]:
import ipywidgets
from ipywidgets import Button, Layout, Textarea, HBox, VBox, Label
import os
import glob
from IPython.display import clear_output
import traceback

l = Layout(flex='0 1 auto', height='100px', min_height='100px', width='auto')
process_widget = ipywidgets.Textarea(description='ログ', value='', layout=l)

process_no = 0
DEBUG = False
def write_log(msg):
    global process_widget, process_no
    process_no = process_no + 1
    log_message = f"{process_no}: {msg}\n"
    process_widget.value = log_message + process_widget.value
    
    # ログファイルに書き込む
    if DEBUG:
        with open("/home/jetson/data/notebooks/logfile.log", "a") as log_file:
            log_file.write(log_message)
        
    # UIのクリアと更新
    clear_output(wait=True)

## PWMの値の読み込み

In [ ]:
import Fabo_PCA9685
import time
import pkg_resources
import smbus
import time
import json

SMBUS='smbus'
BUSNUM=i2c_busnum
SERVO_HZ=60
INITIAL_VALUE=300
bus = smbus.SMBus(BUSNUM)
PCA9685 = Fabo_PCA9685.PCA9685(bus,INITIAL_VALUE,address=0x40)
PCA9685.set_hz(SERVO_HZ)

STEERING_CH = 0
THROTTLE_CH = 1
direction = 0
REVERSE = 0
NORMAL = 1

pwm_front = 0
pwm_back = 0

with open('pwm_params.json') as f:
    json_str = json.load(f)

    pwm_stop = json_str["pwm_speed"]["stop"]
    pwm_front = json_str["pwm_speed"]["front"]
    pwm_back = json_str["pwm_speed"]["back"]
    pwm_left = json_str["pwm_steering"]["left"]
    pwm_center = json_str["pwm_steering"]["center"]
    pwm_right = json_str["pwm_steering"]["right"]


if pwm_front >= pwm_back:
    direction = REVERSE
else:
    direction = NORMAL

PCA9685.set_channel_value(STEERING_CH, pwm_center)
PCA9685.set_channel_value(THROTTLE_CH, pwm_stop)

In [ ]:
try:
    with open('raw_params.json') as f:
        json_str = json.load(f)

        raw_stop = json_str["raw_speed"]["stop"]
        raw_front = json_str["raw_speed"]["front"]
        raw_back = json_str["raw_speed"]["back"]
        raw_left = json_str["raw_steering"]["left"]
        raw_center = json_str["raw_steering"]["center"]
        raw_right = json_str["raw_steering"]["right"]
except:
    print("Don't exit raw_param.json")
    raw_stop = None
    raw_front = None
    raw_back = None
    raw_left = None
    raw_center = None
    raw_right = None
# コントローラー入力読み取り用のI2C初期化
i2c_controller = smbus.SMBus(i2c_busnum)
i2c_controller_addr = 0x08

## LED表示機能

In [ ]:
def init_led():
    global i2c_busnum,i2c
    i2c = smbus.SMBus(i2c_busnum)

def change_color(color):
    global i2c
    colors = {
        'normal': 0x10,
        'red':    0x1a,
        'blue':   0x1b,
        'yellow': 0x1c,
        'green':  0x1d,
        'white':  0x1e,
        'orange': 0x1f,
        'purple': 0x20,
        'lime':   0x21,
        'pink':   0x22,
        'off':    0x30,
        'error1':    0x40,
        'error2':    0x41,
        'error3':    0x42, # red, red, white
        'error4':    0x43, # red, white, red
    }
    if color in colors:
        i2c.read_i2c_block_data(0x08, colors[color])
    else:
        print(f"Error change_color: '{color}' is not a valid color.")

init_led()

change_color("normal")

## カメラの読込

この部分でエラーが発生する場合は、Jetsonの再起動をお願いします。<br>
それでも、カメラが認識できない場合は、ケーブルの接続確認をしてください。


In [ ]:
CAM0_FPS=60
CAM1_FPS=60

In [ ]:
from jetcam.csi_camera import CSICamera
from jetcam.usb_camera import USBCamera
import os
os.environ['OPENCV_LOG_LEVEL'] = 'SILENT'

def open_camera():
    global cam0, cam1
    try:
        cam0 = CSICamera(capture_device=0,width=224, height=224, capture_fps=CAM0_FPS)
        # cam1 = CSICamera(capture_device=1,width=224, height=224, capture_fps=CAM1_FPS)
        cam1 = USBCamera(capture_device=2,width=224, height=224, capture_fps=CAM1_FPS)
    except Exception as e:
        # スタックトレースを含むエラーメッセージを取得
        error_message = f"Error open_camera:{e}\n{''.join(traceback.format_exception(None, e, e.__traceback__))}"
        write_log(error_message)

## クロッピング関数

In [ ]:
import cv2
import numpy as np

def clip_box_px(box, w, h):
    """画像サイズに合わせて (l,t,r,b) をクリップ"""
    l, t, r, b = box
    l = max(0, min(l, w))
    r = max(0, min(r, w))
    t = max(0, min(t, h))
    b = max(0, min(b, h))
    if r <= l: r = min(w, l+1)
    if b <= t: b = min(h, t+1)
    return int(l), int(t), int(r), int(b)

def rel_to_abs(box_rel, w, h):
    """相対座標 (xc,yc,w,h) → ピクセル (l,t,r,b)"""
    xc, yc, bw, bh = box_rel
    l = int((xc - bw/2) * w)
    t = int((yc - bh/2) * h)
    r = int((xc + bw/2) * w)
    b = int((yc + bh/2) * h)
    return clip_box_px((l, t, r, b), w, h)

def mask_keep_roi_and_resize(img, box_px, size=(224, 224)):
    """
    ROI以外を黒で塗りつぶしてから 224x224 にリサイズ
    box_px: (left, top, right, bottom) [ピクセル]
    """
    h, w = img.shape[:2]
    l, t, r, b = clip_box_px(box_px, w, h)
    masked = np.zeros_like(img)
    masked[t:b, l:r] = img[t:b, l:r]
    out = cv2.resize(masked, size, interpolation=cv2.INTER_CUBIC)
    return out

def crop_only_and_resize(img, box_px, size=(224, 224)):
    """
    ROIを切り抜いてから 224x224 にリサイズ（黒ベタなし）
    """
    h, w = img.shape[:2]
    l, t, r, b = clip_box_px(box_px, w, h)
    crop = img[t:b, l:r]
    out = cv2.resize(crop, size, interpolation=cv2.INTER_CUBIC)
    return out

## クラス分類
電光掲示板の矢印を分類。

In [ ]:
from fabo import EnvironmentCategory

# EnvironmentCategory
#
# 開発者ノート:
# CATEGORIES (文字列 -> Enum.name) と POST_* (整数、0ベースインデックス -> Enum.value) 両方を纏めて定義

# CATEGORIES = [
#     "GATE_LEFT",  # 電子掲示板で左矢印表示
#     "GATE_RIGHT",  # 電子掲示板で右矢印表示
#     "GATE_CENTER",  # 電子掲示板で前矢印表示
#     "PARKING",  # 駐車状態
#     "ETC",  # 分類できないもの
# ]

## カテゴリーインデックスの番号

In [ ]:
# POS_LEFT = 0
# POS_RIGHT = 1
# POS_CENTER = 2
# POS_PARKING = 3
# POS_ETC = 4


## 走行処理

In [ ]:
import threading
import torch
from utils import preprocess
import subprocess
import cv2
import time
from torch2trt import TRTModule
import subprocess
import datetime
import torch.nn.functional as F

annotation = False  # 録画におけるAuto Annotationフラグ
record = False  # 録画フラグ
running = False  # AI開始フラグ
running_cam0 = False  # live_cam0() のループ処理継続フラグ
running_cam1 = False  # live_cam0() のループ処理継続フラグ
speed_ai_flag = False  # 速度推論フラグ

def map_rc(x, in_min, in_max, out_min, out_max):
    return (x - in_min) * (out_max - out_min) // (in_max - in_min) + out_min

def handle(x):
    global pwm_right,pwm_left,STEERING_CH,PCA9685
    x = map_rc(x, 224, 0, pwm_right, pwm_left)
    PCA9685.set_channel_value(STEERING_CH, x)

def throttle(speed, front_value, stop_value):
    global pwm_front,pwm_back,THROTTLE_CH,PCA9685
    speed = map_rc(speed, 224, 0, front_value, stop_value)
    PCA9685.set_channel_value(THROTTLE_CH, speed)

IMG_WIDTH=224

In [ ]:
#対応するモデル名を求める。対応するモデル名がない場合は、デフォルトで電光掲示板右側を走行する
def get_model(target):
    if target == "GATE_RIGHT":
        name = "model_right"
        return name,model_a_trt
    elif target == "GATE_LEFT":
        name = "model_left"
        return name,model_b_trt
    elif target == "PARKING":
        name = "model_parking"
        return name,model_c_trt
    elif target == "GATE_CENTER":
        name = "model_center"
        return name,model_d_trt
    else:
        name = "GATE_RIGHT"
        return name,model_a_trt

## カメラ０　周回数カウントアップ、モデル切り替え

In [ ]:
status = 0

def live_cam0():
    """
    カメラ0(フロントカメラ)からの映像を使用して走行制御を行う関数。
    大回り直進、大回り右折、小回り直進、小回り右折の4つのモデルから選べらたモデルを用いて
    ステアリング操作（handle関数呼び出し）とスロットル制御（throttle関数呼び出し）を行います。
    
    カメラ映像の記録を行う機能も備えています。
    """
    global FPS_30, IMG_WIDTH, annotation, cam0, cam0, count_cam0, fps_type, model_a_trt, num, position, preprocess, pwm_stop, record, running_cam0, save_dir0, save_dir1, selected_model, speed_ai_flag, status, target

    try:
        count_cam0 = 1
        num = 1
        frame_count = 0
        # 処理開始時間
        start_time = time.time()
        # 走行用推論実行時間
        process_drive_time = 0
        # 走行用推論実行時間(総計)
        total_process_drive_time = 0

        selected_model = model_a_trt

        last_detect = 0
        last_lap_time = 0
        lap = 0

        final_lap = 0

        speed = 0
        #Road Start!
        position = EnvironmentCategory.GATE_RIGHT

        left_count = 0
        right_count = 0
        center_count = 0

        detect_count = 0

        last_detect_time = 0
        model_name, model = get_model("GATE_RIGHT")
        selectec_model = model
    except Exception as e:
        write_log(f"Error live_cam0 init:{e}")

    # アノテーション用のディレクトリ設定
    save_xy_dir = None
    save_speed_dir = None
    if record and annotation:
        if all(x is not None for x in [raw_left, raw_right, raw_stop, raw_front]):
            # アノテーション有効時はxy/とspeed/のサブディレクトリを使用
            base_dir = os.path.dirname(save_dir0)
            save_xy_dir = save_dir0
            save_speed_dir = os.path.join(base_dir, 'speed')
            os.makedirs(save_xy_dir, exist_ok=True)
            os.makedirs(save_speed_dir, exist_ok=True)
        else:
            write_log("【警告】raw_params.jsonが読み込まれていないため、アノテーションを無効化します。")
            annotation = False

    while running_cam0:
        try:
            # カメラ画像を読込
            img0 = cam0.read()
            if record:
                remarked_img0 = img0.copy()

                if annotation:
                    # アノテーション有効
                    # I2C制御信号からxy/speedを読み取る
                    try:
                        data = i2c_controller.read_i2c_block_data(i2c_controller_addr, 0x01, 12)

                        # xy値の取得
                        xy = data[0] << 24 | data[1] << 16 | data[2] << 8 | data[3]
                        xy = map_rc(xy, raw_left, raw_right, 0, 224)
                        if xy < 0:
                            xy = 0
                        elif xy > 224:
                            xy = 224

                        # speed値の取得
                        speed_val = data[4] << 24 | data[5] << 16 | data[6] << 8 | data[7]
                        speed_val = map_rc(speed_val, raw_stop, raw_front, 0, 224)
                        if speed_val < 0:
                            speed_val = 0
                        elif speed_val > 224:
                            speed_val = 224

                        # ファイル名生成と保存
                        xy_img_name = "{}_{}_{:0=5}.jpg".format(xy, 112, count_cam0)
                        xy_image_path = os.path.join(save_xy_dir, xy_img_name)
                        cv2.imwrite(xy_image_path, remarked_img0)

                        speed_img_name = "{}_{}_{:0=5}.jpg".format(0, speed_val, count_cam0)
                        speed_image_path = os.path.join(save_speed_dir, speed_img_name)
                        cv2.imwrite(speed_image_path, remarked_img0)

                    except Exception as e:
                        write_log(f"アノテーションエラー: {e}")
                else:
                    # アノテーション無効
                    # xy は0埋め、speed は除外
                    name = f"0_0_{count_cam0:0=5}.jpg"
                    image_path0 = os.path.join(save_dir0, name)
                    cv2.imwrite(image_path0, remarked_img0)

            process_drive_time = time.time()
            img0 = preprocess(img0).half()


            # 走行用の推論を実行（y値は使用しない、x値だけを使用する）
            output = selected_model(img0).detach().cpu().numpy().flatten()
            x = float(output[0]) * steering_gain_slider.value
            y = float(output[1])
            x = int(IMG_WIDTH * (x / 2.0 + 0.5))
            y = int(IMG_WIDTH * (y / 2.0 + 0.5))
            handle(x)

            # スロットルも推論する場合
            if speed_ai_flag == False:
                speed = speed_raw_slider.value
                throttle(speed, pwm_front, pwm_stop)
            else:
                #　駐車の場合は、１００だった場合は、１１２にして暴走しない。（ゆっくり走行）
                if lap > 3:
                    speed = float(output[3])
                    speed = int(IMG_WIDTH * (speed / 2.0 + 0.5)) * speed_gain_slider.value
                    #駐車速度調整　--trhottle inqury contol-- breake!
                    if speed > 224:
                        speed = 224
                    stop_value = pwm_stop + speed_low_slider.value
                    if stop_value >= pwm_front:
                        stop_value = pwm_front
                    throttle(speed, pwm_front, stop_value)
                    #else:
                    #    speed = 0
                    #    throttle(speed, pwm_front, pwm_stop)
                else:
                    #　通常走行モードは、停止しそうな速度調整できるようにする。
                    speed = float(output[3])
                    speed = int(IMG_WIDTH * (speed / 2.0 + 0.5)) * speed_gain_slider.value
                    if speed > 224:
                        speed = 224
                    stop_value = pwm_stop + speed_low_slider.value
                    if stop_value >= pwm_front:
                        stop_value = pwm_front
                    throttle(speed, pwm_front, stop_value)

            #model_name, model = get_model(position.name)


            # 走行用の推論実行時間を計測
            total_process_drive_time += time.time() - process_drive_time

            # 現在の時間を取得
            current_time = time.time()
            #write_log(f"Position={position.name}")

            # LAPの判定
            # 左か右,中央をいずれかを検出したら、2回連続で検出したら周回カウントをアップ、その５秒後に判定再開。
            if current_time - last_lap_time > 5:
                if (position is EnvironmentCategory.GATE_LEFT
                        or position is EnvironmentCategory.GATE_RIGHT
                        or position is EnvironmentCategory.GATE_CENTER):
                    #ログに書き出す０または１，２
                    #write_log(f"LAP_Marker_detect posi{position.name}:count{detect_count}:currenttime{current_time}")
                    detect_count += 1
                    #1回でLap数をカウントアップし初期化
                    if detect_count > 3:
                        lap += 1
                        write_log(f"LAP Addition!")
                        detect_count = 0
                        last_lap_time = current_time  # Lapの時間を更新
                else:
                    # ラップカウントアップから５秒経っていないまたは、５回未満で不連続検出なら
                    #write_log(f"LAP Fault!")
                    detect_count = 0


                #モデル切り替え　３周したらパーキングモデルに切り替えるその以外は、方向指示器のそれぞれモデルをチョイスする。
                #Kaisu
                if lap > 3:
                    # パーキングモデルに変更
                    if final_lap == 0:
                        change_color("orange")
                        model_name, model = get_model("PARKING")
                        selectec_model = model
                        selected_model = model_c_trt
                        final_lap = 1
                        write_log(f"Parking!")
                else:
                    #０〜３周の場合Lapの判定と5秒間後に再開
                    #if current_time - last_detect_time > 5:
                    if final_lap == 0:

                        if position is EnvironmentCategory.GATE_LEFT:
                            left_count += 1
                            #model_name, model = get_model("GATE_LEFT")
                            #selectec_model = model
                            write_log(f"Pre-{position.name}")

                            if left_count > 3:
                                model_name, model = get_model("GATE_LEFT")
                                selectec_model = model
                                selected_model = model_b_trt
                                change_color("blue")
                                #write_log(f"Left")
                                #last_detect_time = current_time  # Lapの時間を更新
                                last_detect_time = current_time
                                left_count = 0
                                right_count = 0
                                center_count = 0

                        elif position is EnvironmentCategory.GATE_RIGHT:
                            right_count += 1
                            #model_name, model = get_model("GATE_RIGHT")
                            #selectec_model = model
                            write_log(f"Pre-{position.name}")

                            if right_count > 1:
                                model_name, model = get_model("GATE_RIGHT")
                                selectec_model = model
                                selected_model = model_a_trt
                                change_color("purple")
                                #write_log(f"RIGHT")
                                #last_detect_time = current_time  # Lapの時間を更新
                                last_detect_time = current_time
                                left_count = 0
                                right_count = 0
                                center_count = 0

                        elif position is EnvironmentCategory.GATE_CENTER:
                            center_count += 1
                            #model_name, model = get_model("GATE_CENTER")
                            #selectec_model = model
                            write_log(f"Pre-{position.name}")

                            if center_count > 1:
                                model_name, model = get_model("GATE_CENTER")
                                selectec_model = model
                                selected_model = model_d_trt
                                change_color("yellow")
                                write_log(f"CENTER")
                                #last_detect_time = current_time  # Lapの時間を更新
                                last_detect_time = current_time
                                left_count = 0
                                right_count = 0
                                center_count = 0

            # カウンターを増加
            count_cam0 += 1
            # フレーム用のカウンターを増加
            frame_count += 1
            
            # 走行時の処理時間計測
            if time.time() - start_time > 3.0:
                fps = frame_count / 3.0
                speed_type = ""
                
                write_log(f"Cam0(走行用) FPS: {fps:.1f}, Lap: {lap}, Speed: {speed:.1f} {speed_type}, Steering Gain: {steering_gain_slider.value},  走行推論: {total_process_drive_time/(fps*3)*1000:.1f}ms ")
                frame_count = 0
                start_time = time.time()
                total_process_drive_time = 0
            
        except Exception as e:
            # スタックトレースを含むエラーメッセージを取得
            error_message = f"Error live_cam0:{e}\n{''.join(traceback.format_exception(None, e, e.__traceback__))}"
            write_log(error_message)
            
    if record:
        write_log(f"画像を{count_cam0}枚の走行データを保存しました。")

## カメラ1処理

In [ ]:
last_detect_time = 0  # 最後に検出があった時刻（秒）

def check_non_detection_period(elapsed_time_ms):
    """
    特定の期間内にイベントが検出されていないかどうかをチェックする。
    elapsed_time_msはミリ秒単位で指定する。

    Args:
        elapsed_time_ms (int): 検出がないと判断する期間（ミリ秒）

    Returns:
        bool: 指定された非検出期間を超えていればTrue、そうでなければFalse
    """
    global last_detect_time
    current_time = time.time()  # 現在時刻を取得（秒）

    if last_detect_time == 0:  # 初期状態の場合
        last_detect_time = current_time  # 最初の検出時刻を設定

    period_time_ms = (current_time - last_detect_time) * 1000  # 経過時間をミリ秒に変換

    if period_time_ms > elapsed_time_ms:
        return True  # 指定された非検出期間を超えている
    else:
        return False  # 指定された非検出期間を超えていない

def update_last_detect_time():
    """
    イベント検出時に最後の検出時刻を現在時刻に更新する。
    """
    global last_detect_time
    last_detect_time = time.time()  # 現在時刻を更新（秒）

In [ ]:
def check_stop_time(current_time, target_detected_time, stop_time_ms):
    # target_detected_time および current_time は秒単位であるため、
    # ミリ秒単位での停止時間を判断するには、秒単位の差をミリ秒単位に変換する
    elapsed_time_ms = (current_time - target_detected_time) * 1000
    if elapsed_time_ms >= stop_time_ms:
        return True
    else:
        return False

In [ ]:
def live_cam1():
    """
    カメラ1(サイドカメラ)からの映像を使用して環境認識を行う関数。
    クラス分類して認識した結果から現在の状態を変えていく
    """
    global position,last_detect_time,target, cam1, speed_ai_flag,running_cam1,cam1,IMG_WIDTH,record,model_a_trt,model_b_trt,model_c_trt,model_class_trt,preprocess,pwm_stop,save_dir0,save_dir1,num,count_cam1,fps_type,FPS_30,status

    try:
        # 認識回数
        detect_count = 0
        # 環境情報の認識時間計測
        total_process_detect_time = 0
        # カウンター
        count_cam1 = 1
        # フレーム用カウンター
        frame_count = 0
        start_time = time.time()
        last_detect = 0

        target_detected_time = None

    except Exception as e:
        write_log(f"Error live_cam1 init:{e}")


    # 例：ピクセル指定のROI（左,上,右,下）
    roi_box_px = (50, 90, 174, 115)


    while running_cam1:
        try:
            # サイドカメラ画像の読込
            img1 = cam1.read()

             # 1) ROI以外を黒ベタにしてリサイズ（雲など背景ノイズの影響を最小化）
            img1 = mask_keep_roi_and_resize(img1, roi_box_px, size=(224, 224))

            if record == True:
                remarked_img1 = img1.copy()

            # 環境情報の認識開始時間
            process_detect_time = time.time()
            # 横画像の画像認識は model_class_trtを使用します。
            img1 = preprocess(img1).half()
            output = model_class_trt(img1).detach()
            output = F.softmax(output, dim=1).cpu().numpy().flatten()
            category_index = output.argmax()
            # 環境情報の認識時間
            total_process_detect_time += time.time() - process_detect_time

            # 現在の時間を取得
            current_time = time.time()

            # 位置==環境情報カテゴリを取得
            try:
                position = EnvironmentCategory(category_index)
            except ValueError:
                write_log(f"Unsupported category index `{category_index}`, defaulting to {EnvironmentCategory.GATE_RIGHT.name}")
                position = EnvironmentCategory.GATE_RIGHT

            # 走行のカメラ画像を保存
            if record == True:
                name = "0_0_{:0=5}.jpg".format(count_cam1)
                image_path1 = os.path.join(save_dir1, name)
                cv2.imwrite(image_path1, remarked_img1)

            count_cam1 += 1
            frame_count += 1

            # 走行時の処理時間計測
            if time.time() - start_time > 3.0:
                fps = frame_count / 3.0
                speed_type = ""
                if speed_ai_flag == False:
                    speed_type = f"(固定)"
                else:
                    speed_type = f"(推論)"

                write_log(f"Cam1(環境用) target {target}, detect_count{detect_count}, prebuiling {prebuilding}, target {target}, status {status}, mode {mode}, car_id {car_id}, FPS: {fps:.1f}, 環境推論: {total_process_detect_time/(fps*3)*1000:.1f}ms")
                frame_count = 0
                start_time = time.time()
                total_process_detect_time = 0
        except Exception as e:

            error_message = f"Error live_cam1:{e}\n{''.join(traceback.format_exception(None, e, e.__traceback__))}"
            
    if record == True:
        write_log(f"画像を{count_cam1}枚の走行データを保存しました。")

In [ ]:
import os

from fabo import asset_root

def start_cameras():
    global cam0, cam1, running_cam0, running_cam1, execute_thread_cam0, execute_thread_cam1
    
    open_camera()

    # Cam0を起動
    running_cam0 = True
    execute_thread_cam0 = threading.Thread(target=live_cam0)
    execute_thread_cam0.start()
    write_log("Start cam0")

    # Cam1を起動
    running_cam1 = True
    execute_thread_cam1 = threading.Thread(target=live_cam1)
    execute_thread_cam1.start()
    write_log("Start cam1")

def setup_save_directory(base_path):
    # 指定された基本パスに基づいて保存ディレクトリを作成し、ログに記録します。

    os.makedirs(base_path, exist_ok=True)
    write_log(f"{base_path}にデータを保存します。")

def run(change):
    global mode, running, start_time, save_dir0, save_dir1, load_model_a_trt, load_model_b_trt, load_model_c_trt, load_model_d_trt, load_model_e_trt, load_model_class_trt
    write_log("run")
    if not (load_model_a_trt and load_model_b_trt and load_model_class_trt):
        write_log("モデルが読み込まれていません")
        return
    if not running:
        write_log("AIが起動しました。")
        if record:
            if name_widget.value != "":
                base_path = os.path.join(asset_root(), "camera", name_widget.value)
                # Cam0とCam1の保存先を設定
                save_dir0 = os.path.join(f"{base_path}_cam0", "xy")
                save_dir1 = os.path.join(f"{base_path}_cam1", "xy")
                
                setup_save_directory(save_dir0)
                setup_save_directory(save_dir1)
            else:
                write_log("【Error】 映像の保存先を入力してください。")
                return
        write_log("カメラを起動中...")
        start_cameras()
        start_time = time.time()

        
def stop(change):
    global running_cam0,running_cam1,execute_thread_cam0,execute_thread_cam1,end_time,start_time,count_cam0,pwm_stop,mode_running
    if running_cam0 == True:
        try:
            end_time = time.time() - start_time
            fps = count_cam0/int(end_time)
            process_time = int((end_time/count_cam0)*1000)
        except:
            fps = -1
            process_time = -1
        mode_running = False
        write_log("AIを停止しました。")
        write_log("処理結果:FPS: " + str(round(fps,2)) + ",処理回数: " + str(count_cam0) + ",　処理時間(1回平均値): " + str(process_time) + " ms")
        running = False
        running_cam0 = False
        running_cam1 = False

        #変数初期化
        final_lap = 0
        change_color("off")

        PCA9685.set_channel_value(THROTTLE_CH, pwm_stop)
        try:
            execute_thread_cam0.join()
            execute_thread_cam1.join()
        except:
            write_log("Thread joinでエラー(すでにthreadが存在しない")
        PCA9685.set_channel_value(THROTTLE_CH, pwm_stop)
        try:
            stop_camera(None)
        except:
            write_log("カメラの停止処理でエラー")

    else:
        PCA9685.set_channel_value(THROTTLE_CH, pwm_stop)
        write_log("現在AIは動いていません。")

## UIのプロパティ設定

In [ ]:
model_a_widget = ipywidgets.Dropdown(options=[],description='モデル')
model_a_time_widget = ipywidgets.Label(description='作成日時')
load_a_button = ipywidgets.Button(description='走行モデルを読込み')

model_b_widget = ipywidgets.Dropdown(options=[],description='モデル')
model_b_time_widget = ipywidgets.Label(description='作成日時')
load_b_button = ipywidgets.Button(description='走行モデルを読込み')

model_c_widget = ipywidgets.Dropdown(options=[],description='モデル')
model_c_time_widget = ipywidgets.Label(description='作成日時')
load_c_button = ipywidgets.Button(description='走行モデルを読込み')

model_d_widget = ipywidgets.Dropdown(options=[],description='モデル')
model_d_time_widget = ipywidgets.Label(description='作成日時')
load_d_button = ipywidgets.Button(description='走行モデルを読込み')

model_trt_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px")),

model_class_widget = ipywidgets.Dropdown(options=[],description='モデル')
model_class_time_widget = ipywidgets.Label(description='作成日時')
load_class_button = ipywidgets.Button(description='環境モデルを読込み')

model_class_trt_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px")),

run_button = ipywidgets.Button(description='走行開始')
stop_button = ipywidgets.Button(description='停止')

name_widget = ipywidgets.Text(description='映像の保存名')
record_box = ipywidgets.Checkbox(False, description='録画')
annotation_box = ipywidgets.Checkbox(False, description='Auto Annotation')
speed_gain_slider = ipywidgets.FloatSlider(description='Speed gain', min=0.1, max=3.0, step=0.05, value=0.8, orientation='horizontal')
speed_low_slider = ipywidgets.FloatSlider(description='Speed low', min=0, max=100, step=1, value=1, orientation='horizontal')
speed_raw_slider = ipywidgets.IntSlider(description='Speed raw', min=1, max=224, step=1, value=80, orientation='horizontal')
steering_gain_slider = ipywidgets.FloatSlider(description='Steering gain', min=0.1, max=3.0, step=0.1, value=1.7, orientation='horizontal')
speed_dropbox = ipywidgets.Dropdown(options=["推論値","固定値"], description='Speed')


In [ ]:
import numpy as np

def load_model(widget, model_var_name):
    try:
        write_log(f"{widget.value}の読込を実行します(初回は時間がかかります)。")
        model = TRTModule()
        model.load_state_dict(torch.load(widget.value))
        model(preprocess(np.zeros((224, 224, 3)).astype(np.uint8)))
        write_log(f"{widget.value}の読込に成功しました。")
        globals()[model_var_name] = model  # 成功した場合、グローバル変数にモデルをセット
        load_flag_var_name = f"load_{model_var_name}"
        write_log(f"{load_flag_var_name}の読込に成功しました。")
        globals()[load_flag_var_name] = True  # 対応するフラグをTrueにセット
        get_jetson_nano_memory_usage()
    except Exception as e:
        write_log(f"【Error】 {e} : {widget.value} の読込に失敗しました。")

# モデル読み込み関数を各ボタンのクリックイベントにバインドする例
load_a_button.on_click(lambda change: load_model(model_a_widget, 'model_a_trt'))
load_b_button.on_click(lambda change: load_model(model_b_widget, 'model_b_trt'))
load_c_button.on_click(lambda change: load_model(model_c_widget, 'model_c_trt'))
load_d_button.on_click(lambda change: load_model(model_d_widget, 'model_d_trt'))

load_class_button.on_click(lambda change: load_model(model_class_widget, 'model_class_trt'))

In [ ]:
import os

from fabo import asset_root

def model_list(type):
    try:
        files = glob.glob(os.path.join(asset_root(), type, '*.pth'), recursive=True)

        # 走行用の回帰モデル
        if type == "model_trt":
            model_a_widget.options = files
            model_b_widget.options = files
            model_c_widget.options = files
            model_d_widget.options = files

        # クラス分類モデル
        elif type == "model_class_trt":
            model_class_widget.options = files

    except Exception as e:
        if type == "model_trt":
            model_a_widget.options = []
            model_b_widget.options = []
            model_c_widget.options = []
            model_d_widget.options = []
        elif type == "model_class_trt":
            model_class_widget.options = []
        write_log(f"Error model_list:{e}")

model_trt_refresh_button.on_click(lambda button: model_list("model_trt"))
model_class_trt_refresh_button.on_click(lambda button: model_list("model_class_trt"))

model_list("model_trt")
model_list("model_class_trt")

In [ ]:
def update_model_time_widget(widget, time_widget):
    """指定されたウィジェットのファイル選択が変更された際の処理。ファイルの作成時間を表示ウィジェットに設定する。"""
    file = widget.value
    try:
        ts = os.path.getctime(file)
        d = datetime.datetime.fromtimestamp(ts)
        s = d.strftime('%Y-%m-%d %H:%M:%S')
        time_widget.value = s
    except Exception as e:
        time_widget.value = "Error update_model_time_widget: " + str(e)

# 各モデル選択ウィジェットの変更を監視し、対応する時刻表示ウィジェットを更新
model_a_widget.observe(lambda change: update_model_time_widget(model_a_widget, model_a_time_widget), names='value')
model_b_widget.observe(lambda change: update_model_time_widget(model_b_widget, model_b_time_widget), names='value')
model_c_widget.observe(lambda change: update_model_time_widget(model_c_widget, model_c_time_widget), names='value')
model_d_widget.observe(lambda change: update_model_time_widget(model_d_widget, model_d_time_widget), names='value')

model_class_widget.observe(lambda change: update_model_time_widget(model_class_widget, model_class_time_widget), names='value')

In [ ]:
def on_fixed_value_change(change):
    global speed_ai_flag, speed_gain_slider, speed_raw_slider
    if change['new'] == "固定値":
        speed_gain_slider.layout.visibility = 'hidden'
        speed_low_slider.layout.visibility = 'hidden'
        speed_raw_slider.layout.visibility = 'visible'
        speed_gain_slider.disabled = True
        speed_raw_slider.disabled = False
        speed_ai_flag = False
    elif change['new'] == "推論値":
        speed_gain_slider.layout.visibility = 'visible'
        speed_low_slider.layout.visibility = 'visible'
        speed_raw_slider.layout.visibility = 'hidden'
        speed_gain_slider.disabled = False
        speed_raw_slider.disabled = True
        speed_ai_flag = True

speed_gain_slider.layout.visibility = 'visible'
speed_low_slider.layout.visibility = 'visible'
speed_raw_slider.layout.visibility = 'hidden'
speed_gain_slider.disabled = False
speed_raw_slider.disabled = True
speed_ai_flag = True
speed_dropbox.observe(on_fixed_value_change, names='value')

run_button.on_click(run)
stop_button.on_click(stop)

In [ ]:
def on_video(change):
    global record
    record^=True

def on_annotation(change):
    global annotation
    annotation ^= True

record_box.observe(on_video)
annotation_box.observe(on_annotation)

In [ ]:
import subprocess
import re

used_memory_widget = ipywidgets.IntText(description='Useメモリ', value=1)
total_memory_widget = ipywidgets.IntText(description='全メモリ', value=1)
memory_button = ipywidgets.Button(description='使用メモリ量の取得')

def get_jetson_nano_memory_usage(event=None):
    command = 'tegrastats'
    try:
        process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
        
        mem_usage_pattern = re.compile(r'RAM (\d+)/(\d+)MB')
        
        max_lines_to_read = 10
        for _ in range(max_lines_to_read):
            line = process.stdout.readline()
            if not line:
                break 
            matches = mem_usage_pattern.search(line)
            if matches:
                used_memory_widget.value = int(matches.group(1))
                total_memory_widget.value = int(matches.group(2))
                process.kill()
                return
        
        process.kill()  
        return

    except subprocess.CalledProcessError as e:
        return

get_jetson_nano_memory_usage()
memory_button.on_click(get_jetson_nano_memory_usage)

In [ ]:
import time

release_button = ipywidgets.Button(description='Camera開放')

def stop_camera(c):
    global cam0,cam1
    cam0.running = False
    cam1.running = False
    time.sleep(0.1)
    cam1.cap.release()
    cam0.cap.release()
    write_log("カメラ0,1を開放しました。")

release_button.on_click(stop_camera)

走行までの流れは以下の通りです。

1. <b>走行モデルを指定してLoadする</b><br>
2. <b>環境モデルを指定してLoadする</b><br>
3. <b>Speedに固定値, Speedに推論値のいずれかを選択する</b><br>
固定値を選んだ場合は固定の速度で走り続けます。推論値を選んだ推論結果を速度に反映します。速度用のアノテーションは13_annotation.ipynbで追加可能です。<br>
4. <b>[オプション] 走行動画を録画する場合は、録画にチェックマークをいれて、保存ファイル名を指定する</b><br>
./runフォルダに保存<br>
5. <b>走行開始ボタンを押して、プロポの裏側のボタンを押して AIモードで自動走行開始する</b><br>
6. <b>終了時は、停止ボタンを押す</b><br>
録画のチェックマークがついている場合は、停止で録画も終了<br>
Speed Inferenceのチェックマークがついている場合は、スロットル量の推論も有効になる<br>
<br>
カメラが60fpsで動いている場合は16ms以内、カメラが30fpsで動いている場合は、33ms以内での処理完了が正常な挙動となります。

In [ ]:
separator = ipywidgets.HTML('<hr style="border-color:gray;margin:10px 0"/>')
title1 = ipywidgets.HTML('<b>【1.使用する環境モデルClass】</b> TensoRTに変換済みのモデルをLoadします。')
title2 = ipywidgets.HTML('<b>【2.使用する走行モデルA(Right)】</b> TensoRTに変換済みのモデルをLoadします。')
title3 = ipywidgets.HTML('<b>【3.使用する走行モデルB(Left)】</b> TensoRTに変換済みのモデルをLoadします。')
title4 = ipywidgets.HTML('<b>【4.使用する走行モデルC(Parking)】</b> TensoRTに変換済みのモデルをLoadします。')
title5 = ipywidgets.HTML('<b>【5.使用する走行モデルD(Center)】</b> TensoRTに変換済みのモデルをLoadします。')

title10 = ipywidgets.HTML('<b>【10.Steeringゲイン】</b> Steeringのゲイン調整します。周りが悪い時は値を1.0以上にします。')
title11 = ipywidgets.HTML('<b>【11.速度】</b> 速度は固定値か、推論から反映かが選べます。')
title12 = ipywidgets.HTML('<b>【12.録画】</b> 走行中の映像を録画したい場合はチェックマークを選択し、保存データセット名を指定してください。Auto Annotationにチェックを入れると、制御信号の値もファイル名に含めて保存します。')
title13 = ipywidgets.HTML('<b>【13.走行の開始】</b> 走行開始を押す前にタイヤが空転していのを確認してください。プロポの裏のボタンを変換しAIモードにすると動き始めます。')

data_collection_widget = ipywidgets.VBox([
    separator,
    title1,
    ipywidgets.HBox([model_class_widget, model_class_trt_refresh_button, model_class_time_widget, load_class_button]),
    process_widget,
    separator,
    title2,
    ipywidgets.HBox([model_a_widget, model_trt_refresh_button, model_a_time_widget, load_a_button]),
    process_widget,
    separator,
    title3,
    ipywidgets.HBox([model_b_widget, model_trt_refresh_button, model_b_time_widget, load_b_button]),
    process_widget,
    separator,
    title4,
    ipywidgets.HBox([model_c_widget, model_trt_refresh_button, model_c_time_widget, load_c_button]),
    process_widget,
    separator,
    title5,
    ipywidgets.HBox([model_d_widget, model_trt_refresh_button, model_d_time_widget, load_d_button]),
    process_widget,
    separator,
    title10,
    ipywidgets.HBox([steering_gain_slider]),
    separator,
    title11,
    ipywidgets.HBox([speed_dropbox]),
    ipywidgets.HBox([speed_raw_slider,speed_low_slider, speed_gain_slider]),
    separator,
    title12,
    ipywidgets.HBox([record_box, annotation_box]),
    name_widget,
    separator,
    title13,
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    ipywidgets.HBox([run_button, stop_button]),
    process_widget,
    separator,
])
display(data_collection_widget)